# MEDDIAG Training on Google Colab Free Tier

**Purpose:** Continue Stage 2 LoRA training from checkpoint (step 100+)

**Prerequisites:**
- HuggingFace token (get from https://huggingface.co/settings/tokens)
- Google Drive with checkpoint files uploaded
- ~5 GB free space on Google Drive

**Training Time:** ~4-5 days to completion (Colab will disconnect periodically; use --resume to continue)

---

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted at /content/drive')

## Step 2: Clone Repository & Set Working Directory

In [ ]:
import os
import subprocess

# Clone repo
!cd /content && git clone https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git

# Set working directory
os.chdir('/content/visual-language-model-research-qlora-cot-rag')
print(f'✓ Working directory: {os.getcwd()}')
!pwd

## Step 3: Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers peft bitsandbytes datasets huggingface_hub
!pip install -q faiss-cpu scikit-learn pillow
!pip install -q pytest

print('✓ All dependencies installed')

## Step 4: Set HuggingFace Token

**IMPORTANT:** Paste your HuggingFace token below (get from https://huggingface.co/settings/tokens)

In [ ]:
# REPLACE 'your_hf_token_here' with your actual token
HF_TOKEN = 'your_hf_token_here'

if HF_TOKEN == 'your_hf_token_here':
    raise ValueError('❌ STOP: You must set your HuggingFace token above!')

os.environ['HF_TOKEN'] = HF_TOKEN
print(f'✓ HF_TOKEN set (first 10 chars): {HF_TOKEN[:10]}...')

## Step 5: Copy Checkpoint from Google Drive

**First, upload your checkpoint to Google Drive:**
1. On your local machine, zip the `models/lora_step100/` folder
2. Upload the zip to your Google Drive (e.g., `/My Drive/meddiag_checkpoint.zip`)
3. Then run this cell

**Path format examples:**
- `/content/drive/My Drive/meddiag_checkpoint.zip`
- `/content/drive/Shareddrives/YourDrive/meddiag_checkpoint.zip`

In [ ]:
import zipfile
import shutil

# REPLACE with your actual path to the checkpoint zip on Google Drive
CHECKPOINT_ZIP_PATH = '/content/drive/My Drive/meddiag_checkpoint.zip'

if not os.path.exists(CHECKPOINT_ZIP_PATH):
    print(f'❌ Checkpoint not found at: {CHECKPOINT_ZIP_PATH}')
    print('\nAvailable files in My Drive:')
    !ls -lh '/content/drive/My Drive/' | head -20
else:
    print(f'✓ Found checkpoint: {CHECKPOINT_ZIP_PATH}')
    
    # Extract
    print('Extracting...')
    with zipfile.ZipFile(CHECKPOINT_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('models/')
    
    # Verify
    if os.path.exists('models/lora_step100/adapter_model.safetensors'):
        print('✓ Checkpoint extracted successfully')
        !ls -lh models/lora_step100/
    else:
        print('❌ Extraction failed — check zip structure')

## Step 6: Run Unit Tests (Sanity Check)

In [ ]:
# Quick sanity check
!python -m pytest tests/ --ignore=tests/test_integration.py -q --tb=line 2>&1 | tail -10
print('✓ Tests passed (or warnings only)')

## Step 7: Start Training (Resume from Checkpoint)

⚠️ **IMPORTANT NOTES:**
- Colab sessions last ~12 hours before auto-disconnecting
- Use `--resume` flag — it will pick up from the last checkpoint
- Training logs are saved to `logs/stage2.jsonl` (check progress there)
- To continue after disconnect: re-run this cell (it will resume automatically)

In [ ]:
import subprocess
import time

# Start training
print('Starting Stage 2 training with resume...')
print('This will run until completion or Colab session disconnects.')
print('Check logs/stage2.jsonl for real-time progress.')
print('')

result = subprocess.run(
    'bash run_pipeline.sh --resume',
    shell=True,
    cwd='/content/visual-language-model-research-qlora-cot-rag',
    capture_output=False,
    text=True
)

print(f'\n✓ Training completed with exit code: {result.returncode}')

## Step 8: Monitor Training Progress (Run While Training)

Run this cell periodically while training runs in another cell to see real-time progress.

In [ ]:
import json
import pandas as pd

log_file = 'logs/stage2.jsonl'

if os.path.exists(log_file):
    # Read all log entries
    logs = []
    with open(log_file, 'r') as f:
        for line in f:
            logs.append(json.loads(line))
    
    df = pd.DataFrame(logs)
    
    # Show last 10 steps
    print('=== Last 10 Training Steps ===')
    print(df[['step', 'epoch', 'loss', 'lr', 'vram_gb', 'elapsed_s']].tail(10).to_string(index=False))
    
    # Summary
    latest = logs[-1]
    hours_elapsed = latest['elapsed_s'] / 3600
    print(f'\n=== Training Summary ===')
    print(f'Current step: {latest["step"]} / ~1650 estimated total')
    print(f'Current epoch: {latest["epoch"]} / 3')
    print(f'Latest loss: {latest["loss"]:.4f}')
    print(f'Time elapsed: {hours_elapsed:.1f} hours')
    print(f'VRAM usage: {latest["vram_gb"]:.2f} GB')
    
    # Estimate remaining
    if latest['step'] > 100:
        avg_time_per_step = latest['elapsed_s'] / latest['step']
        remaining_steps = 1650 - latest['step']
        remaining_hours = (remaining_steps * avg_time_per_step) / 3600
        print(f'\nEstimated remaining: {remaining_hours:.1f} hours ({remaining_hours/24:.1f} days)')
else:
    print('❌ Training logs not found. Has training started?')

## Step 9: Save Final Checkpoint to Google Drive

Run this after training completes to save the final model back to Google Drive.

In [ ]:
import shutil

# Zip the final models
print('Zipping final checkpoint...')
shutil.make_archive('meddiag_final_checkpoint', 'zip', 'models', 'lora_adapter')

# Copy to Google Drive
output_path = '/content/drive/My Drive/meddiag_final_checkpoint.zip'
shutil.copy('meddiag_final_checkpoint.zip', output_path)

print(f'✓ Final checkpoint saved to Google Drive: {output_path}')
print(f'File size: {os.path.getsize(output_path) / (1024**2):.1f} MB')

## Troubleshooting

### Colab disconnected while training?
- Click "Runtime" → "Restart runtime"
- Re-run cells 1-5 to remount drive and restore checkpoint
- Re-run cell 7 with `--resume` — it will pick up from last checkpoint

### "HF_TOKEN not set" error?
- Make sure you replaced 'your_hf_token_here' in Step 4
- Get token from https://huggingface.co/settings/tokens

### Out of memory (OOM)?
- Colab free tier has ~12 GB VRAM
- The model uses ~4 GB, should be fine
- If OOM, reduce `--max-pairs` in `run_pipeline.sh` (default 4000)

### Training is very slow?
- Free tier uses K80 GPU (~260s per optimizer step expected)
- Pro tier (T4/A100) is much faster but requires payment
- Use `--resume` to checkpoint and continue later